**TÍTULO**

F3N5 — Ventos zonais e o feedback de Bjerknes

**CONTEXTO**

Bjerknes (1969) descreveu o laço de retroalimentação positiva do ENSO: o enfraquecimento dos alísios reduz o gradiente zonal de SST, que enfraquece a circulação de Walker e os próprios alísios. Quantificar esse acoplamento é central para explicar crescimento e pico dos eventos.

**DESAFIO**

Hipótese: anomalias de vento zonal (tau_x, u850) covariam com o gradiente zonal de SSTA com defasagem de poucas semanas, e a estrutura espacial da correlação vento × SSTA(longitude) apresenta o dipolo característico do acoplamento. Objetivos: correlações e regressões defasadas série × série e série × campo.

**METODOLOGIA**

Séries semanais normalizadas de tau_x_anom e u850_anom (caixa Niño 3.4/Niño 4, F2). Gradiente zonal de SSTA a partir do campo OISST equatorial (média 2°S–2°N): G(t) = SSTA_oeste(170°W–150°W) − SSTA_leste(90°W–80°W). Correlações defasadas (±26 semanas) com N efetivo de Bretherton; regressão espacial SSTA(lon, t) = a(lon)·tau_x(t−lag) + b com significância por t-teste. O domínio OISST local cobre 170°W–30°W; longitudes a oeste, quando exigidas, são declaradas como indisponíveis (sem preenchimento).

**RESULTADOS ESPERADOS**

TabF3N5_correlacao_defasada_vento_gradiente.csv, TabF3N5_regressao_espacial.csv; FigF3N5_lagcorr_vento_gradiente, FigF3N5_regressao_espacial (a(lon) com faixa de significância) e FigF3N5_dispersao_acoplamento (tau_x × gradiente, com ajuste).

**REFERÊNCIAS BIBLIOGRÁFICAS**

1. BJERKNES, J. Atmospheric Teleconnections from the Equatorial Pacific. Monthly Weather Review, v. 97, p. 163-172, 1969. DOI: https://doi.org/10.1175/1520-0493(1969)097<0163:ATFTEP>2.3.CO;2
2. HERSBACH, H. et al. The ERA5 global reanalysis. Quarterly Journal of the Royal Meteorological Society, v. 146, p. 1999-2049, 2020. DOI: https://doi.org/10.1002/qj.3803
3. BRETHERTON, C. S. et al. The Effective Number of Spatial Degrees of Freedom of a Time-Varying Field. Journal of Climate, v. 12, p. 1990-2009, 1999.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists())
if str(RAIZ / 'src') not in sys.path:
    sys.path.insert(0, str(RAIZ / 'src'))
from nino_brasil import fase3_nino as f3

f3.ensure_dirs()
master = f3.load_master_weekly()
fisicas = [c for c in master.columns if c != 'ocean_source_code']
print(f'Master semanal F2: {master.shape[0]} semanas x {len(fisicas)} variaveis fisicas'
      f" | simulado={master.attrs.get('dado_simulado', False)}")
print(f'Periodo: {master.index.min().date()} a {master.index.max().date()}')

Master semanal F2: 2376 semanas x 44 variaveis fisicas | simulado=False
Periodo: 1981-01-04 a 2026-07-12


In [2]:
sst_lon = f3.load_oisst_equatorial_weekly()
ssta_lon = f3.lon_anomaly_matrix(sst_lon)
lons = np.array([float(c) for c in ssta_lon.columns])
oeste = ssta_lon.loc[:, (lons >= -170) & (lons <= -150)].mean(axis=1)
leste = ssta_lon.loc[:, (lons >= -90) & (lons <= -80)].mean(axis=1)
gradiente = (oeste - leste).rename('gradiente_zonal_ssta_c')
tau = f3.zscore(pd.to_numeric(master['tau_x_anom'], errors='coerce'))
u850 = f3.zscore(pd.to_numeric(master['u850_anom'], errors='coerce'))
print(f'campo SSTA: {ssta_lon.shape}; simulado={sst_lon.attrs.get("dado_simulado", False)}')

campo SSTA: (2342, 561); simulado=False


In [3]:
linhas = []
for nome, serie in (('tau_x_anom', tau), ('u850_anom', u850)):
    for lag in range(-26, 27):
        par = pd.concat([serie.shift(lag), gradiente], axis=1).dropna()
        if len(par) < 52:
            continue
        r = float(par.iloc[:, 0].corr(par.iloc[:, 1]))
        n_eff = f3.effective_sample_size(par.iloc[:, 0], par.iloc[:, 1])
        linhas.append({'vento': nome, 'lag_semanas': lag, 'r': r,
                       'n_eff': round(n_eff, 1), 'p_valor': f3.correlation_p_value(r, n_eff)})
lagcorr = pd.DataFrame(linhas)
f3.save_table(lagcorr, 'TabF3N5_correlacao_defasada_vento_gradiente', index=False)

fig, ax = plt.subplots(figsize=(9, 4.5))
for nome, grupo in lagcorr.groupby('vento'):
    ax.plot(grupo['lag_semanas'], grupo['r'], marker='.', label=nome)
    sig = grupo[grupo['p_valor'] < 0.05]
    ax.scatter(sig['lag_semanas'], sig['r'], s=18, zorder=5)
ax.axvline(0, color='k', lw=0.6); ax.axhline(0, color='k', lw=0.6)
ax.set_xlabel('Lag (semanas; negativo = vento lidera o gradiente)')
ax.set_ylabel('r')
ax.set_title('Acoplamento vento zonal x gradiente zonal de SSTA (pontos: p<0,05)')
ax.legend()
fig.tight_layout()
f3.save_figure(fig, 'FigF3N5_lagcorr_vento_gradiente')
plt.close(fig)

[tabela] TabF3N5_correlacao_defasada_vento_gradiente.csv (106x5) -> C:\DEV\NINO26\data\processed\numeric-tables\fase3


[figura] FigF3N5_lagcorr_vento_gradiente.png (+pdf) -> C:\DEV\NINO26\data\processed\figures\fase3


In [4]:
melhor = lagcorr.loc[lagcorr.groupby('vento')['r'].apply(lambda s: s.abs().idxmax())]
lag_otimo = int(melhor.loc[melhor['vento'] == 'tau_x_anom', 'lag_semanas'].iloc[0])
linhas = []
for coluna in ssta_lon.columns:
    par = pd.concat([tau.shift(lag_otimo), ssta_lon[coluna]], axis=1).dropna()
    if len(par) < 52:
        continue
    x = par.iloc[:, 0].to_numpy(); y = par.iloc[:, 1].to_numpy()
    a, b = np.polyfit(x, y, 1)
    r = float(np.corrcoef(x, y)[0, 1])
    n_eff = f3.effective_sample_size(par.iloc[:, 0], par.iloc[:, 1])
    linhas.append({'lon': float(coluna), 'coef_regressao_c_por_sigma_tau': a,
                   'r': r, 'p_valor': f3.correlation_p_value(r, n_eff)})
regressao = pd.DataFrame(linhas)
f3.save_table(regressao, 'TabF3N5_regressao_espacial', index=False)

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(regressao['lon'], regressao['coef_regressao_c_por_sigma_tau'], lw=1.2)
sig = regressao[regressao['p_valor'] < 0.05]
ax.scatter(sig['lon'], sig['coef_regressao_c_por_sigma_tau'], s=10, color='crimson', label='p<0,05')
ax.axhline(0, color='k', lw=0.6)
ax.set_xlabel('Longitude (graus; dominio OISST local 170W-30W)')
ax.set_ylabel('dSSTA/dtau_x (C por sigma)')
ax.set_title('Regressao espacial SSTA(lon) sobre tau_x (feedback de Bjerknes)')
ax.legend()
fig.tight_layout()
f3.save_figure(fig, 'FigF3N5_regressao_espacial')
plt.close(fig)

par = pd.concat([tau.shift(lag_otimo).rename('tau_x_z'), gradiente], axis=1).dropna()
a, b = np.polyfit(par['tau_x_z'], par['gradiente_zonal_ssta_c'], 1)
fig, ax = plt.subplots(figsize=(6.5, 6))
ax.scatter(par['tau_x_z'], par['gradiente_zonal_ssta_c'], s=6, alpha=0.4)
xx = np.linspace(par['tau_x_z'].min(), par['tau_x_z'].max(), 50)
ax.plot(xx, a * xx + b, color='crimson', label=f'ajuste: {a:.3f} C/sigma (lag {lag_otimo} sem)')
ax.set_xlabel('tau_x_anom (z)'); ax.set_ylabel('gradiente zonal de SSTA (C)')
ax.set_title('Dispersao do acoplamento oceano-atmosfera')
ax.legend()
fig.tight_layout()
f3.save_table(par, 'FigF3N5_dispersao_acoplamento_dados')
f3.save_figure(fig, 'FigF3N5_dispersao_acoplamento')
plt.close(fig)

[tabela] TabF3N5_regressao_espacial.csv (447x4) -> C:\DEV\NINO26\data\processed\numeric-tables\fase3


[figura] FigF3N5_regressao_espacial.png (+pdf) -> C:\DEV\NINO26\data\processed\figures\fase3
[tabela] FigF3N5_dispersao_acoplamento_dados.csv (2322x2) -> C:\DEV\NINO26\data\processed\numeric-tables\fase3


[figura] FigF3N5_dispersao_acoplamento.png (+pdf) -> C:\DEV\NINO26\data\processed\figures\fase3
